In [ ]:
import requests
import pandas as pd
from datetime import datetime

# -----------------------------
# CONFIG
# -----------------------------
PATENTVIEW_URL = "https://api.patentsview.org/patents/query"
GDELT_DOC_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

# -----------------------------
# 1. GET PATENTS FOR A MONTH
# -----------------------------
def get_patents(year: int, month: int, limit=50):
    start_date = f"{year}-{month:02d}-01"
    
    query = {
        "q": {
            "patent_date": {
                "$gte": start_date,
                "$lte": f"{year}-{month:02d}-31"
            }
        },
        "f": [
            "patent_number",
            "patent_title",
            "patent_date",
            "patent_abstract"
        ],
        "o": {"per_page": limit}
    }

    r = requests.post(PATENTVIEW_URL, json=query)
    data = r.json()

    return data.get("patents", [])


# -----------------------------
# 2. NEWS MENTIONS (GDELT)
# -----------------------------
def get_news_mentions(query, start_date, end_date):
    params = {
        "query": query,
        "mode": "ArtList",
        "format": "json",
        "startdatetime": start_date + "000000",
        "enddatetime": end_date + "235959",
        "maxrecords": 250,
        "sort": "HybridRel"
    }

    try:
        r = requests.get(GDELT_DOC_URL, params=params, timeout=10)
        data = r.json()
        return len(data.get("articles", []))
    except Exception:
        return 0


# -----------------------------
# 3. BUILD BUZZ DATASET
# -----------------------------
def build_buzz_report(year, month):
    patents = get_patents(year, month)

    results = []

    start_date = f"{year}{month:02d}01000000"
    end_date = f"{year}{month:02d}31235959"

    for p in patents:
        title = p.get("patent_title", "")
        abstract = p.get("patent_abstract", "")

        # simple search query (title works best)
        query = title[:120]

        mentions = get_news_mentions(query, start_date, end_date)

        # simple "buzz score"
        buzz_score = mentions

        results.append({
            "patent_number": p.get("patent_number"),
            "title": title,
            "date": p.get("patent_date"),
            "news_mentions": mentions,
            "buzz_score": buzz_score
        })

    df = pd.DataFrame(results)
    df = df.sort_values(by="buzz_score", ascending=False)

    return df


# -----------------------------
# 4. RUN
# -----------------------------
if __name__ == "__main__":
    year = 2025
    month = 12  # change this

    df = build_buzz_report(year, month)

    print("\nTOP TALKED ABOUT PATENTS:\n")
    print(df.head(10))

    df.to_csv(f"patent_buzz_{year}_{month}.csv", index=False)